# 13 — THEME attention first derivative vs anchor ETF price

Notebook 12's velocity view, through the THEME lens: how fast is attention
on each theme accelerating or collapsing, drawn against the theme's anchor
ETF. The derivative (change vs the previous period) leads the raw counts —
this is the "speed" chart the take-off detection runs on.

Themes with a young anchor ETF (IBIT, MAGS, EUAD...) plot from the ETF's
first real price date, with a marker showing where price data begins.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
P = os.path.join(ROOT, 'data', 'processed')
PRICES_PATH = os.path.join(ROOT, 'data', 'prices', 'prices.parquet')

# window: the PIPELINE_* env vars (set by update_data.py, including its
# --start/--end overrides) win; otherwise fall back to the constants at the
# top of update_data.py. Same toggle as live vs backtest either way.
import update_data
START_DATE = os.environ.get('PIPELINE_START_DATE') or update_data.START_DATE
END_DATE = os.environ.get('PIPELINE_END_DATE')
if END_DATE is None:
    END_DATE = update_data.END_DATE
WIN_LO = pd.to_datetime(START_DATE)
WIN_HI = pd.to_datetime(END_DATE) if END_DATE else None
print('window:', WIN_LO.date(), '->', (WIN_HI.date() if WIN_HI is not None else 'LIVE (newest)'))

def clip_series(s):
    s = s[s.index >= WIN_LO]
    return s if WIN_HI is None else s[s.index <= WIN_HI]

def clip_dates(df, col):
    df = df[df[col] >= WIN_LO]
    return df if WIN_HI is None else df[df[col] <= WIN_HI]

def load_prices():
    if not os.path.exists(PRICES_PATH):
        raise FileNotFoundError('prices.parquet not found - run  python pull_bloomberg_prices.py  first.')
    px = pd.read_parquet(PRICES_PATH); px['date'] = pd.to_datetime(px['date'])
    return px

def price_series(prices, symbol):
    # daily close, then made CONTINUOUS (forward-fill weekends/holidays) so the
    # line is smooth with no gaps. Clip to the window.
    one = prices[prices['symbol'] == symbol].sort_values('date')
    s = one.set_index('date')['px_last']
    if not s.empty:
        s = s.asfreq('D').ffill()
    return clip_series(s)


# --- x-axis tick control (X_TICKS in the parameters cell) ---
# 'auto' = matplotlib decides; 'W' = a label every week; 'M' = every month.
# Weekly labels are only readable on windows up to ~6 months - use
# PLOT_LAST_DAYS to zoom in first.
import matplotlib.dates as mdates

def set_date_ticks(ax, ticks):
    if ticks == 'W':
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))  # Mondays
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %y'))
    elif ticks == 'M':
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')

In [ ]:
HOW_MANY = 6      # how many top themes to plot
# DEFAULTS TUNED FOR THE TRADING VIEW: the last 1-2 months, day by day.
# For a multi-year regime view use FREQ='W', PLOT_LAST_DAYS=None, DERIV_SMOOTH=4.
FREQ = 'D'        # 'D' daily (high fidelity), 'W' weekly, 'M' monthly
WINDOW = 7        # daily rolling window the derivative is taken over
X_TICKS = 'W'     # x-axis labels: 'auto', 'W' (weekly), 'M' (monthly)
PLOT_LAST_DAYS = 60     # zoom to the last N days (None = whole window)
NORMALISE = True  # derivative of the mention SHARE (%), not raw counts - share of all THEME mentions
MIN_TOTAL = 30    # mask days with fewer total theme mentions than this
DERIV_SMOOTH = 5  # moving average over this many periods (days here) - short
                  # enough to react within a week, long enough to kill the
                  # day-to-day sawtooth

In [ ]:
from src.themes import THEME_ETFS

counts = pd.read_parquet(os.path.join(P, 'daily_theme_counts.parquet'))
# aggregates carry 'mention_count'; older notebook-04 output carries
# keyword/inferred columns - handle both
if 'mention_count' not in counts.columns:
    counts['mention_count'] = (counts.get('keyword_count', 0)
                               + counts.get('inferred_count', 0))
counts['date'] = pd.to_datetime(counts['date']); counts = clip_dates(counts, 'date')
if PLOT_LAST_DAYS:                       # zoom: only the most recent stretch
    WIN_LO = max(WIN_LO, counts['date'].max() - pd.Timedelta(days=PLOT_LAST_DAYS))
    counts = counts[counts['date'] >= WIN_LO]
day_totals = counts.groupby('date')['mention_count'].sum()

themes = counts.groupby('theme')['mention_count'].sum().sort_values(ascending=False).head(HOW_MANY).index.tolist()
print('auto themes (most mentioned):', themes)
prices = load_prices()

for theme in themes:
    etf = THEME_ETFS.get(theme)
    if etf is None:
        print('skip', theme, '- no anchor ETF'); continue
    m = (counts[counts['theme'] == theme].sort_values('date')
         .set_index('date')['mention_count'].asfreq('D').fillna(0))
    if NORMALISE:
        totals = day_totals.reindex(m.index).fillna(0)
        m = (m / totals.replace(0, pd.NA)) * 100
        m[totals < MIN_TOTAL] = pd.NA
    px = price_series(prices, etf)
    all_px = prices[prices['symbol'] == etf]
    price_starts = all_px['date'].min() if len(all_px) else None
    if FREQ == 'D':
        base = m.rolling(WINDOW, min_periods=1).mean() if NORMALISE else m.rolling(WINDOW).sum()
        deriv = base.diff()
    else:
        base = m.resample(FREQ).mean() if NORMALISE else m.resample(FREQ).sum()
        deriv = base.diff()
        if not px.empty:
            px = px.resample(FREQ).last()
    deriv_smooth = deriv.rolling(DERIV_SMOOTH, min_periods=1).mean()

    fig, ax1 = plt.subplots(figsize=(13, 5))
    ax1.axhline(0, color='black', linewidth=0.6)
    ax1.plot(deriv.index, deriv.values, color='tab:green', linewidth=0.9,
             alpha=0.35, label='1st derivative (raw)')
    ax1.plot(deriv_smooth.index, deriv_smooth.values, color='tab:green',
             linewidth=2.2, label=f'{DERIV_SMOOTH}-period moving average')
    ax1.legend(loc='upper left')
    ax1.set_ylabel(('change in theme share (pp)' if NORMALISE else f'change in mentions per {FREQ}'),
                   color='tab:green')
    ax1.tick_params(axis='y', labelcolor='tab:green')
    if px.empty:
        note = (f'no {etf} price in this window'
                + (f' ({etf} starts {price_starts.date()})' if price_starts is not None else ''))
        ax1.set_title(f'{theme}: attention first derivative ({note})')
        print('note', theme, '-', note)
    else:
        ax2 = ax1.twinx()
        ax2.plot(px.index, px.values, color='tab:red', linewidth=1.8, label='price')
        ax2.set_ylabel('price (USD)', color='tab:red'); ax2.tick_params(axis='y', labelcolor='tab:red')
        if price_starts is not None and (px.index.min() - WIN_LO).days > 7:
            ax1.axvline(px.index.min(), color='gray', linestyle='--', linewidth=1)
            ax1.annotate(f'{etf} price data starts', (px.index.min(), ax1.get_ylim()[1]),
                         fontsize=8, ha='left', va='top', color='gray')
            print('note', theme, f'- {etf} price data only starts {px.index.min().date()}')
        ax1.set_title(f'{theme}: attention first derivative vs {etf} price ({FREQ})')
    ax1.grid(True, alpha=0.3)
    set_date_ticks(ax1, X_TICKS)
    fig.tight_layout(); plt.show()